# Why is revenue up but profit not?
Walkthrough notebook. **Run `python python/run_all.py` first** (from the repo root) so `data/processed/` exists, then run all cells and commit the notebook *with outputs* so GitHub renders it.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROC = Path('..') / 'data' / 'processed'
pd.options.display.float_format = '{:,.2f}'.format
sns.set_theme(style='whitegrid')
df = pd.read_csv(PROC / 'orders_clean.csv', parse_dates=['order_date'])
df.shape

## 1. Revenue vs profit

In [ ]:
yearly = pd.read_csv(PROC / 'yearly_kpis.csv')
yearly[['order_year','revenue','profit','margin_pct','rev_growth_pct','profit_growth_pct']]

In [ ]:
monthly = pd.read_csv(PROC / 'monthly_kpis.csv', parse_dates=['order_month'])
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(monthly.order_month, monthly.revenue, label='Revenue')
ax.plot(monthly.order_month, monthly.profit, label='Profit')
ax2 = ax.twinx(); ax2.plot(monthly.order_month, monthly.margin_pct, 'r--', label='Margin %'); ax2.grid(False)
ax.legend(loc='upper left'); ax.set_title('Revenue, profit and margin by month'); plt.show()

## 2. Hypothesis 1: discount erosion

In [ ]:
bands = pd.read_csv(PROC / 'discount_band_profit.csv')
display(bands[['discount_band','lines','revenue','profit','margin_pct','loss_line_pct']])
plt.bar(bands.discount_band, bands.margin_pct, color=['#2ca02c' if m>=0 else '#d62728' for m in bands.margin_pct])
plt.axhline(0, color='k'); plt.title('Margin % by discount band'); plt.show()

In [ ]:
hi = df[df.discount > 0.20]
print(f"Lines >20% discount: {hi.sales.sum()/df.sales.sum():.1%} of revenue, profit ${hi.profit.sum():,.0f}, margin {hi.profit.sum()/hi.sales.sum():.1%}")

## 3. Hypothesis 2: mix shift and the profit bridge

In [ ]:
sub = pd.read_csv(PROC / 'subcategory_profit.csv').sort_values('profit')
sub[['category','sub_category','revenue','profit','margin_pct','avg_discount']].head(8)

In [ ]:
bridge = pd.read_csv(PROC / 'profit_bridge_total.csv')
bridge

In [ ]:
last = pd.read_csv(PROC / 'profit_bridge_subcategory.csv')
last = last[last.year_to == last.year_to.max()]
last.sort_values('margin_effect')[['sub_category','delta_sales','growth_effect','margin_effect']].head(8)

## 4. Hypotheses 3 and 4: returns and regions

In [ ]:
ret = (df.drop_duplicates('order_id').groupby('category').returned.mean() * 100).round(2)
print('Order return rate by category (%)'); print(ret)
state = pd.read_csv(PROC / 'state_profit.csv')
state[state.profit < 0][['region','state','revenue','profit','margin_pct','avg_discount']]

## 5. Hypothesis 5: customers, RFM and cohorts

In [ ]:
seg = pd.read_csv(PROC / 'rfm_segment_summary.csv')
seg[['segment','customers','revenue_share_pct','profit_share_pct','margin_pct','profit_per_customer']]

In [ ]:
cust = pd.read_csv(PROC / 'customer_summary.csv')
cust.groupby('customer_type').agg(customers=('customer_id','count'), profit=('profit','sum'),
                                  first_order_discount=('first_order_discount','mean'))

In [ ]:
ann = pd.read_csv(PROC / 'cohort_retention_annual.csv')
mat = ann.pivot(index='cohort_year', columns='period', values='retention')
sns.heatmap(mat, annot=True, fmt='.0%', cmap='Blues', vmin=0, vmax=1)
plt.title('Annual cohort retention'); plt.show()

## 6. Conclusion
# Key findings (auto-generated)

Every number below was computed by `python/04_profit_drivers.py` from your data.
Use them to fill in `insights/why_profit_lags_revenue.md`.

## 1. Revenue vs profit by year

| Year | Revenue | Profit | Margin | Revenue growth | Profit growth |
|---|---|---|---|---|---|
| 2011 | $179,068 | $19,020 | 10.6% |  |  |
| 2012 | $163,954 | $19,708 | 12.0% | -8.4% | 3.6% |
| 2013 | $261,514 | $43,667 | 16.7% | 59.5% | 121.6% |
| 2014 | $280,924 | $27,953 | 10.0% | 7.4% | -36.0% |

## 2. Profit bridge (growth effect vs margin effect)

| Period | Δ Revenue | Δ Profit | Growth effect | Margin effect |
|---|---|---|---|---|
| 2011→2012 | -$15,114 | $688 | -$2,379 | $3,067 |
| 2012→2013 | $97,560 | $23,959 | $15,383 | $8,576 |
| 2013→2014 | $19,409 | -$15,714 | -$1,139 | -$14,575 |

Biggest margin erosion in the latest year (2014), by sub-category:

- **Machines**: margin effect -$6,775 on revenue change of $2,033
- **Tables**: margin effect -$3,802 on revenue change of $5,022
- **Bookcases**: margin effect -$1,386 on revenue change of -$2,940
- **Phones**: margin effect -$919 on revenue change of -$2,627
- **Chairs**: margin effect -$897 on revenue change of $4,022

## 3. Discount erosion

- Loss-making line items: **18.6%** of all lines, destroying **$61,764** of profit.
- Overall margin: **12.5%**. Excluding loss-making lines it would be **24.3%**.

| Discount band | Lines | Revenue | Profit | Margin |
|---|---|---|---|---|
| 0% | 1,874 | $418,534 | $125,677 | 30.0% |
| 1-20% | 1,557 | $336,836 | $38,924 | 11.6% |
| 21-40% | 172 | $84,330 | -$13,951 | -16.5% |
| 40%+ | 371 | $45,760 | -$40,303 | -88.1% |

- Lines discounted above 20% are **14.7%** of revenue but contribute **-$54,254** of profit (margin -41.7%).

Share of revenue sold at >20% discount, by year: 2011: 18.6%, 2012: 12.5%, 2013: 11.0%, 2014: 16.9%

## 4. Product mix

Bottom 5 sub-categories by total profit:

- **Tables** (Furniture): profit -$6,253, margin -7.0%, avg discount 25.2%
- **Bookcases** (Furniture): profit -$1,215, margin -2.9%, avg discount 21.5%
- **Supplies** (Office Supplies): profit -$959, margin -7.5%, avg discount 10.2%
- **Fasteners** (Office Supplies): profit $298, margin 25.0%, avg discount 8.9%
- **Envelopes** (Office Supplies): profit $2,267, margin 41.4%, avg discount 8.3%

Top 5 sub-categories by total profit:

- **Copiers** (Technology): profit $23,795, margin 38.9%
- **Accessories** (Technology): profit $17,529, margin 25.8%
- **Phones** (Technology): profit $16,572, margin 12.9%
- **Paper** (Office Supplies): profit $12,892, margin 43.6%
- **Chairs** (Furniture): profit $11,694, margin 9.7%

Category share of revenue by year (mix shift check):

| Year | Furniture | Office Supplies | Technology |
|---|---|---|---|
| 2011 | 35.7% | 29.9% | 34.4% |
| 2012 | 36.9% | 27.5% | 35.6% |
| 2013 | 29.4% | 26.0% | 44.6% |
| 2014 | 30.4% | 32.7% | 37.0% |

## 5. Returns

No Returns sheet found in the raw file, so returns analysis was skipped.

## 6. Regional drag

- **9** of 47 states are loss-making, losing **$40,488** in total.

Five worst states:

- **Texas** (Central): revenue $77,075, profit -$12,948, avg discount 36.8%
- **Illinois** (Central): revenue $25,906, profit -$9,089, avg discount 38.9%
- **Pennsylvania** (East): revenue $39,214, profit -$5,797, avg discount 32.2%
- **North Carolina** (South): revenue $30,622, profit -$4,512, avg discount 29.5%
- **Tennessee** (South): revenue $7,616, profit -$2,517, avg discount 31.0%

## 7. Customer quality

- **One-time** customers: 160 (21.9%), profit $9,947 (9.0% of total), avg first-order discount 15.6%
- **Repeat** customers: 569 (78.1%), profit $100,401 (91.0% of total), avg first-order discount 15.4%
- Top 10% of customers by profit generate **71.7%** of total profit.
- Bottom 10% of customers by profit (decile 10) total **-$33,394**.

RFM segments:

| Segment | Customers | Revenue share | Profit share | Margin |
|---|---|---|---|---|
| Champions | 176 | 37.9% | 36.1% | 11.9% |
| Loyal | 140 | 17.5% | 13.3% | 9.5% |
| New | 61 | 4.1% | 4.4% | 13.2% |
| Needs Attention | 60 | 5.4% | 5.9% | 13.6% |
| At Risk | 121 | 24.1% | 30.4% | 15.8% |
| Lost | 171 | 11.0% | 9.9% | 11.2% |
